# Functional Annotation working examples

I will annotate detected modules using three layers of evidence:

1. I will use local CORUM overlap and enrichment analysis to identify whether a module corresponds to known protein complexes. 
2. I will query g:Profiler to obtain broader functional annotations from GO, Reactome, KEGG, and related databases. 
3. I will use an LLM as an interpretation layer, summarizing statistically supported annotations into readable biological explanations. This separates evidence generation from language-based explanation and reduces the risk of over-interpreting the modules.

In [1]:
import os
import sys
import json
import warnings
import pandas as pd
import numpy as np
from scipy.stats import hypergeom
from statsmodels.stats.multitest import multipletests
from pathlib import Path

In [2]:
notebook_dir = os.getcwd()

# Adjust the number of ".." based on where your notebook lives relative to the project root
PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, ".."))
print(f"Project root determined as: {PROJECT_ROOT}")
MODULES_ROOT = os.path.join(PROJECT_ROOT, "outputs", "network", "modules")
CORUM_PATH   = os.path.join(PROJECT_ROOT, "data", "raw", "CORUM_data.txt")
OUT_DIR      = os.path.join(PROJECT_ROOT, "outputs", "network", "annotation")
os.makedirs(OUT_DIR, exist_ok=True)

METHODS   = ["autoencoder_cosine", "composite_rbf_jaccard", "combined_pearson_jaccard"]
STRATEGY  = "top10perNodes"
TOP_N     = 5
FDR_CUTOFF = 0.05

Project root determined as: /Users/aaroncui/Desktop/PPI


For functional annotation, I retrieved human related pairs

In [3]:
print("Loading CORUM complexes...")
corum_raw = pd.read_csv(CORUM_PATH, sep="\t")
corum_raw = corum_raw[corum_raw["Organism"] == "Human"].reset_index(drop=True)

corum_db = {}
for _, row in corum_raw.iterrows():
    genes_str = str(row["subunits.Gene.name."]).strip()
    if not genes_str or genes_str == "nan":
        continue
    genes = set(g.strip() for g in genes_str.split(";") if g.strip())
    if len(genes) >= 2:
        corum_db[row["ComplexID"]] = {
            "name": row["ComplexName"],
            "genes": genes,
            "size": len(genes),
        }

print(f"  Loaded {len(corum_db)} CORUM complexes (human, >=2 subunits)")


Loading CORUM complexes...
  Loaded 3528 CORUM complexes (human, >=2 subunits)


In [4]:
print("Selecting annotation targets...")

records = []
for method in METHODS:
    base = os.path.join(MODULES_ROOT, method, STRATEGY)

    # Background: all proteins in the graph
    assignments = pd.read_csv(os.path.join(base, "module_assignments.csv"))
    background  = set(assignments["protein"].astype(str).str.strip())

    # Top 5 Leiden modules by size
    mod_summary = pd.read_csv(os.path.join(base, "module_summary.csv"))
    top_leiden  = mod_summary.sort_values("size", ascending=False).head(TOP_N)

    for rank, (_, row) in enumerate(top_leiden.iterrows(), 1):
        module_proteins = set(
            assignments[assignments["module_id"] == row["module_id"]]["protein"]
            .astype(str).str.strip()
        )
        records.append({
            "method":        method,
            "strategy":      STRATEGY,
            "cluster_type":  "leiden",
            "cluster_id":    row["module_id"],
            "rank":          rank,
            "size":          row["size"],
            "key_proteins":  row["key_proteins"],
            "proteins":      ";".join(sorted(module_proteins)),
            "n_background":  len(background),
            "background":    ";".join(sorted(background)),
        })

    # Top 5 MCODE cores by mcode_score
    cores   = pd.read_csv(os.path.join(base, "mcode_cores.csv"))
    members = pd.read_csv(os.path.join(base, "mcode_core_members.csv"))
    top_mc  = cores.sort_values("mcode_score", ascending=False).head(TOP_N)

    for rank, (_, row) in enumerate(top_mc.iterrows(), 1):
        core_proteins = set(
            members[members["core_id"] == row["core_id"]]["protein"]
            .astype(str).str.strip()
        )
        records.append({
            "method":        method,
            "strategy":      STRATEGY,
            "cluster_type":  "mcode",
            "cluster_id":    row["core_id"],
            "rank":          rank,
            "size":          row["size"],
            "key_proteins":  row["key_proteins"],
            "proteins":      ";".join(sorted(core_proteins)),
            "n_background":  len(background),
            "background":    ";".join(sorted(background)),
        })

targets_df = pd.DataFrame(records)
# targets_df.to_csv(os.path.join(OUT_DIR, "selected_annotation_targets.csv"), index=False)
print(f"  {len(targets_df)} annotation targets saved")

targets_df.head()

Selecting annotation targets...
  30 annotation targets saved


,method,strategy,cluster_type,cluster_id,rank,size,key_proteins,proteins,n_background,background
0,autoencoder_cosine,top10perNodes,leiden,L0001,1,664,CSK;MAP3K3;CAP1;ACTR2;GRB2;ACTR3;ARHGAP30;STK4...,ABI1;ACAP1;ACAP2;ACAP3;ACLY;ACTA1;ACTBL2;ACTN4...,12142,A1BG;A1CF;A2M;A2ML1;A4GALT;AAAS;AACS;AADAT;AAE...
1,autoencoder_cosine,top10perNodes,leiden,L0002,2,617,NCF2;SIGLEC9;FGR;TLR8;FGD2;ARHGEF10L;HVCN1;EVI...,ABCD1;ABHD6;ABR;ABTB1;ACBD4;ACBD5;ACP5;ACSL3;A...,12142,A1BG;A1CF;A2M;A2ML1;A4GALT;AAAS;AACS;AADAT;AAE...
2,autoencoder_cosine,top10perNodes,leiden,L0003,3,612,DDOST;MLEC;RPN2;PRKCSH;FKBP2;GANAB;EDEM2;STT3A...,ABCD3;ABCD4;ABHD11;ABHD12;ABHD13;ABTB2;ACAA1;A...,12142,A1BG;A1CF;A2M;A2ML1;A4GALT;AAAS;AACS;AADAT;AAE...
3,autoencoder_cosine,top10perNodes,leiden,L0004,4,559,DDX17;SREK1;TRA2B;SMARCA2;IK;DMAP1;SAFB;SF3B2;...,ABRAXAS1;ACIN1;ACTL6A;ACTR5;ADAR;ADAT1;ADNP;AF...,12142,A1BG;A1CF;A2M;A2ML1;A4GALT;AAAS;AACS;AADAT;AAE...
4,autoencoder_cosine,top10perNodes,leiden,L0005,5,461,PDHX;PDHB;PDHA1;POLG2;ATP5F1B;POLG;CCDC127;VDA...,AARS2;ABCB7;ABCC1;ABHD10;ACAA2;ACAD8;ACADM;ACA...,12142,A1BG;A1CF;A2M;A2ML1;A4GALT;AAAS;AACS;AADAT;AAE...


In [5]:
# ---------------------------------------------------------------------------
# Helper: parse protein list from target row
# ---------------------------------------------------------------------------
def get_proteins(row):
    return [p.strip() for p in str(row["proteins"]).split(";") if p.strip()]

def get_background(row):
    return set(p.strip() for p in str(row["background"]).split(";") if p.strip())

# ---------------------------------------------------------------------------
# 3. CORUM hypergeometric enrichment
# ---------------------------------------------------------------------------
print("Running CORUM hypergeometric enrichment...")

corum_rows = []
for _, tgt in targets_df.iterrows():
    query_genes = set(get_proteins(tgt))
    bg_genes    = get_background(tgt)
    M = len(bg_genes)

    for cid, cinfo in corum_db.items():
        complex_in_bg = cinfo["genes"] & bg_genes
        n = len(complex_in_bg)
        if n < 2:
            continue

        overlap = query_genes & complex_in_bg
        k = len(overlap)
        N = len(query_genes)

        if k == 0:
            continue

        pval = hypergeom.sf(k - 1, M, n, N)

        corum_rows.append({
            "method":        tgt["method"],
            "strategy":      tgt["strategy"],
            "cluster_type":  tgt["cluster_type"],
            "cluster_id":    tgt["cluster_id"],
            "rank":          tgt["rank"],
            "complex_id":    cid,
            "complex_name":  cinfo["name"],
            "complex_size":  cinfo["size"],
            "complex_in_bg": n,
            "query_size":    N,
            "overlap":       k,
            "pval":          pval,
            "overlap_genes": ";".join(sorted(overlap)),
        })

corum_df = pd.DataFrame(corum_rows)
if len(corum_df) > 0:
    # BH FDR correction within each cluster
    fdr_vals = []
    for _, grp in corum_df.groupby(["method", "cluster_type", "cluster_id"]):
        _, fdr, _, _ = multipletests(grp["pval"].values, method="fdr_bh")
        fdr_vals.extend(fdr.tolist())
    corum_df["fdr"] = fdr_vals
    corum_df = corum_df.sort_values(["method", "cluster_type", "cluster_id", "pval"])
    corum_df.to_csv(os.path.join(OUT_DIR, "corum_enrichment_selected.csv"), index=False)
    sig = (corum_df["fdr"] < FDR_CUTOFF).sum()
    print(f"  {len(corum_df)} CORUM tests, {sig} significant (FDR<{FDR_CUTOFF})")
else:
    corum_df.to_csv(os.path.join(OUT_DIR, "corum_enrichment_selected.csv"), index=False)
    print("  No CORUM overlaps found")

corum_df.head()

Running CORUM hypergeometric enrichment...
  6120 CORUM tests, 1765 significant (FDR<0.05)


,method,strategy,cluster_type,cluster_id,rank,complex_id,complex_name,complex_size,complex_in_bg,query_size,overlap,pval,overlap_genes,fdr
571,autoencoder_cosine,top10perNodes,leiden,L0001,1,8561,"TNFR1 signaling complex, TNF-induced",24,23,664,15,3.269353e-14,CHUK;CYLD;IKBKB;IKBKE;IKBKG;MAP3K7;RBCK1;RIPK1...,2.089117e-11
336,autoencoder_cosine,top10perNodes,leiden,L0001,1,6344,"TNFR1 signaling complex, native",15,14,664,10,1.836567e-10,CHUK;IKBKB;IKBKG;MAP3K7;RBCK1;RIPK1;RNF31;TAB1...,5.867832e-08
236,autoencoder_cosine,top10perNodes,leiden,L0001,1,5230,CHUK-NFKB2-REL-IKBKG-SPAG9-NFKB1-NFKBIE-COPB2-...,12,12,664,8,3.124397e-08,CHUK;IKBKG;NFKB1;NFKBIA;NFKBIE;REL;TNIP1;TNIP2,6.654965e-06
0,autoencoder_cosine,top10perNodes,leiden,L0001,1,27,Arp2/3 protein complex,7,7,664,6,1.747417e-07,ACTR2;ACTR3;ARPC1B;ARPC2;ARPC3;ARPC5,2.791499e-05
227,autoencoder_cosine,top10perNodes,leiden,L0001,1,5193,"TNF-alpha/NF-kappa B signaling complex (CHUK, ...",12,12,664,7,8.822896e-07,CHUK;IKBKG;NFKB1;NFKBIA;NFKBIE;REL;TNIP2,9.396384e-05


In [6]:
# ---------------------------------------------------------------------------
# 4. g:Profiler enrichment
# ---------------------------------------------------------------------------
print("Running g:Profiler enrichment...")

try:
    from gprofiler import GProfiler
    gp = GProfiler(return_dataframe=True)
    gp_available = True
except ImportError:
    print("  WARNING: gprofiler-official not installed. Skipping g:Profiler.")
    gp_available = False

gp_rows = []
if gp_available:
    SOURCES = ["GO:BP", "GO:CC", "GO:MF", "REAC", "KEGG", "WP", "CORUM"]
    for idx, tgt in targets_df.iterrows():
        query_genes = get_proteins(tgt)
        bg_genes    = list(get_background(tgt))
        label = f"{tgt['method']} / {tgt['cluster_type']} {tgt['cluster_id']}"
        # print(f"  g:Profiler [{idx+1}/{len(targets_df)}]: {label} ({len(query_genes)} proteins)")

        try:
            result = gp.profile(
                organism="hsapiens",
                query=query_genes,
                background=bg_genes,
                sources=SOURCES,
                significance_threshold_method="fdr",
                user_threshold=FDR_CUTOFF,
                no_evidences=False,
            )
            if len(result) > 0:
                result = result.copy()
                result["method"]       = tgt["method"]
                result["strategy"]     = tgt["strategy"]
                result["cluster_type"] = tgt["cluster_type"]
                result["cluster_id"]   = tgt["cluster_id"]
                result["rank"]         = tgt["rank"]
                gp_rows.append(result)
        except Exception as e:
            print(f"    ERROR: {e}")

if gp_rows:
    gp_df = pd.concat(gp_rows, ignore_index=True)
    # Keep relevant columns
    keep_cols = ["method", "strategy", "cluster_type", "cluster_id", "rank",
                 "source", "native", "name", "p_value", "significant",
                 "term_size", "query_size", "intersection_size", "recall",
                 "precision", "intersections"]
    keep_cols = [c for c in keep_cols if c in gp_df.columns]
    gp_df = gp_df[keep_cols].sort_values(
        ["method", "cluster_type", "cluster_id", "p_value"]
    )
    gp_df.to_csv(os.path.join(OUT_DIR, "gprofiler_enrichment_selected.csv"), index=False)
    print(f"  {len(gp_df)} g:Profiler terms (FDR<{FDR_CUTOFF})")
else:
    pd.DataFrame().to_csv(os.path.join(OUT_DIR, "gprofiler_enrichment_selected.csv"), index=False)
    print("  No significant g:Profiler terms")
    gp_df = pd.DataFrame()

gp_df.head()

Running g:Profiler enrichment...
  14767 g:Profiler terms (FDR<0.05)


,method,strategy,cluster_type,cluster_id,rank,source,native,name,p_value,significant,term_size,query_size,intersection_size,recall,precision,intersections
0,autoencoder_cosine,top10perNodes,leiden,L0001,1,GO:CC,GO:0005829,cytosol,6.074982e-67,True,4521,649,462,0.102190,0.711864,"[ABI1, ACLY, ACTA1, ACTR1B, ACTR2, ACTR3, ADD3..."
1,autoencoder_cosine,top10perNodes,leiden,L0001,1,GO:BP,GO:0141124,intracellular signaling cassette,1.854845e-30,True,1348,649,183,0.135757,0.281972,"[ACTN4, AIF1, AKAP13, ARAF, ARF6, ARFGEF2, ARH..."
2,autoencoder_cosine,top10perNodes,leiden,L0001,1,GO:BP,GO:0016192,vesicle-mediated transport,6.817935e-30,True,1235,649,172,0.139271,0.265023,"[ACAP2, AIF1, ANKRD13A, ANKRD13D, AP1AR, AP1B1..."
3,autoencoder_cosine,top10perNodes,leiden,L0001,1,GO:CC,GO:0005737,cytoplasm,3.258060e-28,True,8807,649,593,0.067333,0.913713,"[ABI1, ACAP1, ACAP2, ACAP3, ACLY, ACTA1, ACTBL..."
4,autoencoder_cosine,top10perNodes,leiden,L0001,1,KEGG,KEGG:05131,Shigellosis,7.847110e-26,True,198,649,59,0.297980,0.090909,"[ACTN4, ACTR2, ACTR3, ACTR3B, ARF6, ARPC1B, AR..."


In [7]:
# ---------------------------------------------------------------------------
# 5. Annotation summary table
# ---------------------------------------------------------------------------
print("Building annotation summary table...")

summary_rows = []
for _, tgt in targets_df.iterrows():
    row_dict = {
        "method":        tgt["method"],
        "strategy":      tgt["strategy"],
        "cluster_type":  tgt["cluster_type"],
        "cluster_id":    tgt["cluster_id"],
        "rank":          tgt["rank"],
        "size":          tgt["size"],
        "key_proteins":  tgt["key_proteins"],
    }

    # CORUM hits
    c_hits = corum_df[
        (corum_df["method"]       == tgt["method"]) &
        (corum_df["cluster_type"] == tgt["cluster_type"]) &
        (corum_df["cluster_id"]   == tgt["cluster_id"]) &
        (corum_df["fdr"]          < FDR_CUTOFF)
    ] if len(corum_df) > 0 else pd.DataFrame()

    row_dict["n_corum_hits"]      = len(c_hits)
    row_dict["top_corum_complex"] = c_hits["complex_name"].iloc[0] if len(c_hits) > 0 else ""
    row_dict["top_corum_pval"]    = c_hits["pval"].iloc[0]         if len(c_hits) > 0 else np.nan
    row_dict["top_corum_fdr"]     = c_hits["fdr"].iloc[0]          if len(c_hits) > 0 else np.nan
    row_dict["top_corum_overlap"] = c_hits["overlap"].iloc[0]      if len(c_hits) > 0 else 0

    # g:Profiler hits per source
    if len(gp_df) > 0:
        gp_hits = gp_df[
            (gp_df["method"]       == tgt["method"]) &
            (gp_df["cluster_type"] == tgt["cluster_type"]) &
            (gp_df["cluster_id"]   == tgt["cluster_id"])
        ]
        for src in ["GO:BP", "GO:CC", "GO:MF", "REAC", "KEGG", "WP", "CORUM"]:
            src_hits = gp_hits[gp_hits["source"] == src] if "source" in gp_hits.columns else pd.DataFrame()
            src_key  = src.lower().replace(":", "_")
            row_dict[f"n_{src_key}"]   = len(src_hits)
            row_dict[f"top_{src_key}"] = src_hits["name"].iloc[0] if len(src_hits) > 0 else ""
    else:
        for src in ["GO:BP", "GO:CC", "GO:MF", "REAC", "KEGG", "WP", "CORUM"]:
            src_key = src.lower().replace(":", "_")
            row_dict[f"n_{src_key}"]   = 0
            row_dict[f"top_{src_key}"] = ""

    summary_rows.append(row_dict)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUT_DIR, "annotation_summary_selected.csv"), index=False)
print(f"  Annotation summary saved ({len(summary_df)} rows)")
summary_df.head()

Building annotation summary table...
  Annotation summary saved (30 rows)


,method,strategy,cluster_type,cluster_id,rank,size,key_proteins,n_corum_hits,top_corum_complex,top_corum_pval,...,n_go_mf,top_go_mf,n_reac,top_reac,n_kegg,top_kegg,n_wp,top_wp,n_corum,top_corum
0,autoencoder_cosine,top10perNodes,leiden,L0001,1,664,CSK;MAP3K3;CAP1;ACTR2;GRB2;ACTR3;ARHGAP30;STK4...,163,"TNFR1 signaling complex, TNF-induced",3.269353e-14,...,167,kinase activity,285,Signal Transduction,115,Shigellosis,182,B cell receptor signaling,90,TNF-R1-associated signaling complex
1,autoencoder_cosine,top10perNodes,leiden,L0002,2,617,NCF2;SIGLEC9;FGR;TLR8;FGD2;ARHGEF10L;HVCN1;EVI...,18,NADPH oxidase,1.917715e-06,...,53,molecular transducer activity,55,Immune System,60,Lysosome biogenesis,24,Microglia pathogen phagocytosis pathway,13,DNA ligase IV-XRCC4-AHNK complex
2,autoencoder_cosine,top10perNodes,leiden,L0003,3,612,DDOST;MLEC;RPN2;PRKCSH;FKBP2;GANAB;EDEM2;STT3A...,41,HRD1 complex,2.456789e-11,...,101,hexosyltransferase activity,90,Asparagine N-linked glycosylation,23,Metabolic pathways,20,N glycan biosynthesis,30,HRD1 complex
3,autoencoder_cosine,top10perNodes,leiden,L0004,4,559,DDX17;SREK1;TRA2B;SMARCA2;IK;DMAP1;SAFB;SF3B2;...,383,"Spliceosome, A complex",4.227443e-67,...,164,nucleic acid binding,188,Processing of Capped Intron-Containing Pre-mRNA,11,Spliceosome,29,mRNA processing,348,CORUM root
4,autoencoder_cosine,top10perNodes,leiden,L0005,5,461,PDHX;PDHB;PDHA1;POLG2;ATP5F1B;POLG;CCDC127;VDA...,28,"F1F0-ATPase, mitochondrial",1.532222e-14,...,113,catalytic activity,78,Aerobic respiration and respiratory electron t...,38,Metabolic pathways,51,Disorders of mitochondrial homeostatis dynamic...,18,"F1F0-ATP synthase, mitochondrial"


In [8]:
# ---------------------------------------------------------------------------
# 6. Method comparison table
# ---------------------------------------------------------------------------
print("Building method comparison table...")

method_rows = []
for method in METHODS:
    sub = summary_df[summary_df["method"] == method]
    leiden_sub = sub[sub["cluster_type"] == "leiden"]
    mcode_sub  = sub[sub["cluster_type"] == "mcode"]

    def safe_mean(s):
        return round(s.mean(), 2) if len(s) > 0 else np.nan

    def safe_sum(s):
        return int(s.sum()) if len(s) > 0 else 0

    method_rows.append({
        "method":                    method,
        "leiden_n_corum_total":      safe_sum(leiden_sub["n_corum_hits"]),
        "leiden_pct_with_corum":     round(100 * (leiden_sub["n_corum_hits"] > 0).mean(), 1) if len(leiden_sub) > 0 else 0,
        "leiden_mean_gobp":          safe_mean(leiden_sub["n_go_bp"]),
        "leiden_mean_reac":          safe_mean(leiden_sub["n_reac"]),
        "leiden_mean_kegg":          safe_mean(leiden_sub["n_kegg"]),
        "mcode_n_corum_total":       safe_sum(mcode_sub["n_corum_hits"]),
        "mcode_pct_with_corum":      round(100 * (mcode_sub["n_corum_hits"] > 0).mean(), 1) if len(mcode_sub) > 0 else 0,
        "mcode_mean_gobp":           safe_mean(mcode_sub["n_go_bp"]),
        "mcode_mean_reac":           safe_mean(mcode_sub["n_reac"]),
        "mcode_mean_kegg":           safe_mean(mcode_sub["n_kegg"]),
        "total_sig_terms":           safe_sum(sub[[c for c in sub.columns if c.startswith("n_") and c not in ["n_corum_hits"]]].sum(axis=1)),
    })

method_cmp = pd.DataFrame(method_rows)
method_cmp.to_csv(os.path.join(OUT_DIR, "method_annotation_comparison.csv"), index=False)
print(f"  Method comparison saved ({len(method_cmp)} rows)")

method_cmp.head()


Building method comparison table...
  Method comparison saved (3 rows)


,method,leiden_n_corum_total,leiden_pct_with_corum,leiden_mean_gobp,leiden_mean_reac,leiden_mean_kegg,mcode_n_corum_total,mcode_pct_with_corum,mcode_mean_gobp,mcode_mean_reac,mcode_mean_kegg,total_sig_terms
0,autoencoder_cosine,633,100.0,494.6,139.2,49.4,41,80.0,44.6,48.8,5.4,6305
1,composite_rbf_jaccard,492,100.0,302.6,87.4,19.0,0,0.0,118.2,36.8,10.6,4740
2,combined_pearson_jaccard,532,100.0,191.0,80.8,13.6,67,80.0,78.6,35.8,13.4,3722


In [9]:
print("Building Leiden vs MCODE comparison table...")

compare_rows = []
for method in METHODS:
    for cluster_type in ["leiden", "mcode"]:
        sub = summary_df[
            (summary_df["method"]       == method) &
            (summary_df["cluster_type"] == cluster_type)
        ]
        if len(sub) == 0:
            continue

        n_cols = [c for c in sub.columns if c.startswith("n_") and c != "n_corum_hits"]
        total_terms = sub[n_cols].sum().sum() if len(n_cols) > 0 else 0

        compare_rows.append({
            "method":              method,
            "cluster_type":        cluster_type,
            "n_clusters":          len(sub),
            "mean_size":           round(sub["size"].mean(), 1),
            "n_corum_total":       int(sub["n_corum_hits"].sum()),
            "pct_with_any_corum":  round(100 * (sub["n_corum_hits"] > 0).mean(), 1),
            "total_gp_terms":      int(total_terms),
            "mean_gp_per_cluster": round(total_terms / len(sub), 1) if len(sub) > 0 else 0,
            "mean_gobp":           round(sub["n_go_bp"].mean(), 1) if "n_go_bp" in sub.columns else 0,
            "mean_reac":           round(sub["n_reac"].mean(), 1) if "n_reac" in sub.columns else 0,
        })

lm_cmp = pd.DataFrame(compare_rows)
lm_cmp.to_csv(os.path.join(OUT_DIR, "leiden_vs_mcode_comparison.csv"), index=False)
print(f"  Leiden vs MCODE comparison saved ({len(lm_cmp)} rows)")


lm_cmp.head()


Building Leiden vs MCODE comparison table...
  Leiden vs MCODE comparison saved (6 rows)


,method,cluster_type,n_clusters,mean_size,n_corum_total,pct_with_any_corum,total_gp_terms,mean_gp_per_cluster,mean_gobp,mean_reac
0,autoencoder_cosine,leiden,5,582.6,633,100.0,5512,1102.4,494.6,139.2
1,autoencoder_cosine,mcode,5,17.0,41,80.0,793,158.6,44.6,48.8
2,composite_rbf_jaccard,leiden,5,707.4,492,100.0,3567,713.4,302.6,87.4
3,composite_rbf_jaccard,mcode,5,19.6,0,0.0,1173,234.6,118.2,36.8
4,combined_pearson_jaccard,leiden,5,595.6,532,100.0,2712,542.4,191.0,80.8


## LLM Explain

In [11]:
llm_interpretations = pd.read_csv(os.path.join(OUT_DIR, "llm_interpretations_openai.csv"))
llm_interpretations.head()

,method,strategy,cluster_type,cluster_id,rank,size,key_proteins,biological_identity,key_evidence,surprising_signals,confidence,confidence_reason,hypothesis
0,autoencoder_cosine,top10perNodes,leiden,L0001,1,664,CSK;MAP3K3;CAP1;ACTR2;GRB2;ACTR3;ARHGAP30;STK4...,TNFR1 signaling complex,"TNFR1 signaling complex, TNF-induced (CORUM) |...",Presence of Arp2/3 protein complex and actin c...,Moderate,Strong enrichment for TNFR1 signaling complex ...,Investigate the potential cross-talk between T...
1,autoencoder_cosine,top10perNodes,leiden,L0002,2,617,NCF2;SIGLEC9;FGR;TLR8;FGD2;ARHGEF10L;HVCN1;EVI...,Innate immune response and NADPH oxidase compl...,NADPH oxidase | overlap=5 | p=1.92e-06 | FDR=6...,Presence of DNA repair complex and transcripti...,Moderate,Strong evidence for immune-related functions b...,Investigate the role of NADPH oxidase in modul...
2,autoencoder_cosine,top10perNodes,leiden,L0003,3,612,DDOST;MLEC;RPN2;PRKCSH;FKBP2;GANAB;EDEM2;STT3A...,Oligosaccharyltransferase complex involved in ...,Oligosaccharyltransferase complex (Stt3A varia...,Presence of generic metabolic pathways and gly...,High,"Strong and consistent evidence across CORUM, G...",Investigate the role of the oligosaccharyltran...
3,autoencoder_cosine,top10perNodes,leiden,L0004,4,559,DDX17;SREK1;TRA2B;SMARCA2;IK;DMAP1;SAFB;SF3B2;...,Spliceosome complex,"Spliceosome, A complex (CORUM) | nucleoplasm (...",Presence of ATP-dependent chromatin remodeling...,High,Strong and consistent evidence across multiple...,Investigate the role of chromatin remodeling p...
4,autoencoder_cosine,top10perNodes,leiden,L0005,5,461,PDHX;PDHB;PDHA1;POLG2;ATP5F1B;POLG;CCDC127;VDA...,Mitochondrial respiratory chain and energy met...,"F1F0-ATPase, mitochondrial | mitochondrion | A...",Presence of aminoacyl-tRNA biosynthesis and va...,High,Strong and consistent evidence across multiple...,Investigate the impact of mitochondrial dysfun...
